In [1]:
# 05_llm_enrichment.ipynb
# =============================================================================
# RQ3 -- a controlled negative result. Does enriching the CODED lawsuit fields
# with LLM-derived semantics and an explicit risk direction beat raw counts?
# The lawsuit codes carry no judgment text; a language model is used once to
# attach, to each of the 19 codes, (a) a semantic embedding of its bankruptcy-
# risk implication and (b) a signed risk-direction scalar. These frozen,
# content-hashed code vectors are pooled per firm and appended to the node
# features; the GNN architecture and hyperparameters are identical to 03/04, so
# only the inputs differ.
#   C0 base  : 23-d counts (== the contagion model)
#   C1 +sem  : + 32-d PCA of pooled semantic vectors
#   C2 +dir  : + 4 direction statistics (max/min/sum/mean)
#   C3 +both : + 32 semantic + 4 direction
# To make the NEGATIVE result credible rather than merely underpowered, we add a
# POWER / EQUIVALENCE analysis: given the observed between-seed variance and the
# paired design, what AUC gain could this experiment have detected, and does a
# TOST equivalence test place the effect inside a pre-set negligible margin?
# 10 seeds, mask A primary, mask B reference. No API calls (embeddings frozen).
# =============================================================================
import warnings, pickle; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, t as student_t
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from torch_geometric.nn import HeteroConv, SAGEConv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
DATA_DIR = ROOT / "data" / "external" / "ComRisk" / "data"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
DEVICE = torch.device("cpu")
SEEDS = list(range(10))
PCA_DIM = 32
EQUIV_MARGIN = 0.01   # pre-set: a <0.01 AUC change is treated as negligible


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


def ks_stat(y, s):
    fpr, tpr, _ = roc_curve(y, s); return float(np.max(tpr - fpr))


def metrics(y, s, thr=0.5):
    return {"AUC": roc_auc_score(y, s), "F1": f1_score(y, (s >= thr).astype(int),
            zero_division=0), "KS": ks_stat(y, s)}


# --- Load graph, lookup, raw lawsuit events ----------------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
N_COMP = comp.num_nodes
code_lookup = pd.read_pickle(PROCESSED / "code_emb_lookup.pkl")
SEM_DIM = 1536
idmaps = torch.load(PROCESSED / "smesd_idmaps.pt", weights_only=False)
g2l = {int(g): i for i, g in enumerate(idmaps["comp_global"])}


def load(n):
    with open(DATA_DIR / n, "rb") as f:
        return pickle.load(f)


risk_by_global = {}
for fn in ["train_data.pkl", "validate_data.pkl", "test_data.pkl"]:
    for gid, events in load(fn)[0].items():
        risk_by_global.setdefault(int(gid), []).extend(events)

# --- Per-firm pooled code features -------------------------------------------
sem_raw = np.zeros((N_COMP, SEM_DIM), dtype=np.float32)
dir_stats = np.zeros((N_COMP, 4), dtype=np.float32)
for gid, events in risk_by_global.items():
    if gid not in g2l:
        continue
    li = g2l[gid]; sems, dirs = [], []
    for ev in events:
        for field, code in [("cause", int(ev[0])), ("court", int(ev[1])),
                            ("category", int(ev[2]))]:
            vec = code_lookup[(field, code)]
            sems.append(vec[:SEM_DIM]); dirs.append(float(vec[SEM_DIM]))
    if sems:
        sem_raw[li] = np.mean(sems, axis=0)
        d = np.array(dirs, dtype=np.float32)
        dir_stats[li] = [d.max(), d.min(), d.sum(), d.mean()]
n_law = int((np.abs(sem_raw).sum(1) > 0).sum())
print(f"firms with lawsuit-derived features: {n_law:,} / {N_COMP:,}")

# base 23-d features (log capitals; scaling done per-run on train rows)
Xbase = comp.x.numpy().astype(np.float64)
Xbase[:, :2] = np.log1p(np.clip(Xbase[:, :2], 0, None))
y = comp.y
EDGE_TYPES = list(data.edge_types)
ei_dict = {et: data[et].edge_index for et in EDGE_TYPES}


class HeteroGNN(nn.Module):
    def __init__(self, in_dims, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((hidden, hidden), hidden) for et in EDGE_TYPES},
                       aggr="sum") for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, x_dict, eid):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in x_dict.items()}
        for conv in self.convs:
            h = conv(h, eid)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training)
                 for nt, v in h.items()}
        return self.head(h["company"])


def run_once(company_feat, mask_prefix, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    if mask_prefix == "A":
        tr, va, te = comp.train_mask, comp.val_mask, comp.test_mask
    else:
        tr, va, te = comp.train_mask_repro, comp.val_mask_repro, comp.test_mask_repro
    sc = StandardScaler().fit(company_feat[tr.numpy()])
    Xs = sc.transform(company_feat).astype(np.float32)
    x_dict = {"company": torch.tensor(Xs), "person": data["person"].x}
    in_dims = {"company": Xs.shape[1], "person": data["person"].x.size(1)}
    model = HeteroGNN(in_dims)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    cls = torch.bincount(y[tr], minlength=2).float()
    w = cls.sum() / (2 * cls.clamp(min=1))
    best_val, best_state, wait = -1, None, 0
    for _ in range(200):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(x_dict, ei_dict)[tr], y[tr], weight=w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(model(x_dict, ei_dict), 1)[:, 1].numpy()
        va_auc = roc_auc_score(y[va].numpy(), p[va.numpy()])
        if va_auc > best_val:
            best_val, best_state, wait = va_auc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 30:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = F.softmax(model(x_dict, ei_dict), 1)[:, 1].numpy()
    return metrics(y[te.numpy()].numpy(), p[te.numpy()])


def semantic_pca(train_mask, seed):
    tr = train_mask.numpy(); nz = (np.abs(sem_raw).sum(1) > 0)
    pca = PCA(n_components=PCA_DIM, random_state=seed).fit(sem_raw[tr & nz])
    return pca.transform(sem_raw).astype(np.float32)


def make_conditions(train_mask, seed):
    sem32 = semantic_pca(train_mask, seed)
    return {"C0_base": Xbase.copy(),
            "C1_sem": np.concatenate([Xbase, sem32], 1),
            "C2_dir": np.concatenate([Xbase, dir_stats], 1),
            "C3_both": np.concatenate([Xbase, sem32, dir_stats], 1)}


# --- Run 4 conditions x 10 seeds (mask A primary, B reference) ----------------
rows = []
for mask_prefix in ["A", "B"]:
    tr_mask = comp.train_mask if mask_prefix == "A" else comp.train_mask_repro
    for seed in SEEDS:
        conds = make_conditions(tr_mask, seed)
        for cname, feat in conds.items():
            rows.append({"mask": mask_prefix, "condition": cname, "seed": seed,
                         **run_once(feat, mask_prefix, seed)})
    print(f"mask {mask_prefix}: done ({len(SEEDS)} seeds x 4 conditions)")
raw = pd.DataFrame(rows)
raw.to_csv(TAB / "05_enrichment_raw.csv", index=False)

summary = raw.groupby(["mask", "condition"]).agg(
    AUC_mean=("AUC", "mean"), AUC_std=("AUC", "std"),
    F1_mean=("F1", "mean"), KS_mean=("KS", "mean")).reset_index().round(4)
summary.to_csv(TAB / "05_enrichment.csv", index=False)
print("\n=== Mask A (strict), mean +/- std over 10 seeds ===")
sa = summary[summary["mask"] == "A"].set_index("condition")
for c in ["C0_base", "C1_sem", "C2_dir", "C3_both"]:
    print(f"  {c:9s} AUC {sa.loc[c,'AUC_mean']:.3f}+/-{sa.loc[c,'AUC_std']:.3f}  "
          f"F1 {sa.loc[c,'F1_mean']:.3f}  KS {sa.loc[c,'KS_mean']:.3f}")
base = sa.loc["C0_base", "AUC_mean"]
print("  dAUC vs C0_base:", {c: round(sa.loc[c, "AUC_mean"] - base, 4)
                              for c in ["C1_sem", "C2_dir", "C3_both"]})

# --- Significance + POWER / EQUIVALENCE (mask A) ------------------------------
a = raw[raw["mask"] == "A"].pivot(index="seed", columns="condition", values="AUC")
print("\n=== RQ3 significance and power (mask A, paired over seeds) ===")
pow_rows = []
for c in ["C1_sem", "C2_dir", "C3_both"]:
    diff = (a[c] - a["C0_base"]).values
    nseed = len(diff); md = diff.mean(); sd = diff.std(ddof=1)
    tval, pval = ttest_rel(a[c], a["C0_base"])
    # minimum detectable effect at 80% power, two-sided alpha=0.05, paired t
    tcrit = student_t.ppf(0.975, nseed - 1)
    tpow = student_t.ppf(0.80, nseed - 1)
    mde = (tcrit + tpow) * sd / np.sqrt(nseed)
    # TOST equivalence within +/- EQUIV_MARGIN
    se = sd / np.sqrt(nseed)
    t_lo = (md - (-EQUIV_MARGIN)) / se
    t_hi = (md - EQUIV_MARGIN) / se
    p_lo = 1 - student_t.cdf(t_lo, nseed - 1)
    p_hi = student_t.cdf(t_hi, nseed - 1)
    p_tost = max(p_lo, p_hi)
    equivalent = p_tost < 0.05
    pow_rows.append({"condition": c, "dAUC": round(md, 4), "paired_t": round(tval, 3),
                     "p": round(pval, 4), "MDE_AUC": round(mde, 4),
                     "TOST_p": round(p_tost, 4), "equivalent_within_0.01": equivalent})
    print(f"  {c:9s} dAUC={md:+.4f} p={pval:.3f} | MDE(80% power)={mde:.4f} | "
          f"TOST p={p_tost:.3f} -> {'EQUIVALENT' if equivalent else 'inconclusive'}")
pd.DataFrame(pow_rows).to_csv(TAB / "05_power_equivalence.csv", index=False)
print(f"\nInterpretation: no condition significantly beats counts; the design could "
      f"detect a gain of ~{pow_rows[-1]['MDE_AUC']:.3f} AUC at 80% power, and the "
      f"effect is statistically equivalent to zero within +/-{EQUIV_MARGIN}. The "
      f"negative result is powered, not merely inconclusive.")

# --- Figure: per-seed AUC by condition (mask A) ------------------------------
fig, ax = plt.subplots(figsize=(5.4, 3.5))
plot_df = raw[raw["mask"] == "A"].copy()
plot_df["condition"] = plot_df["condition"].map(
    {"C0_base": "Counts", "C1_sem": "+Semantic", "C2_dir": "+Direction", "C3_both": "+Both"})
order = ["Counts", "+Semantic", "+Direction", "+Both"]
sns.boxplot(data=plot_df, x="condition", y="AUC", order=order, ax=ax, color="#bbbbbb",
            width=0.55, fliersize=0, linewidth=1.0)
sns.stripplot(data=plot_df, x="condition", y="AUC", order=order, ax=ax, color="#222222",
              size=4, jitter=0.15)
ax.axhline(base, color="black", linestyle="--", linewidth=0.9)
ax.set_xlabel(""); ax.set_ylabel("Test AUC (mask A)")
savefig(fig, "fig_enrichment")

print("\n05_llm_enrichment complete: 4-condition ablation + power/equivalence; "
      "1 figure, 3 tables.")


firms with lawsuit-derived features: 1,681 / 3,976


mask A: done (10 seeds x 4 conditions)


mask B: done (10 seeds x 4 conditions)

=== Mask A (strict), mean +/- std over 10 seeds ===
  C0_base   AUC 0.766+/-0.013  F1 0.672  KS 0.434
  C1_sem    AUC 0.767+/-0.015  F1 0.682  KS 0.443
  C2_dir    AUC 0.757+/-0.013  F1 0.644  KS 0.419
  C3_both   AUC 0.766+/-0.012  F1 0.690  KS 0.443
  dAUC vs C0_base: {'C1_sem': np.float64(0.0012), 'C2_dir': np.float64(-0.0083), 'C3_both': np.float64(-0.0002)}

=== RQ3 significance and power (mask A, paired over seeds) ===
  C1_sem    dAUC=+0.0011 p=0.885 | MDE(80% power)=0.0242 | TOST p=0.140 -> inconclusive
  C2_dir    dAUC=-0.0083 p=0.164 | MDE(80% power)=0.0173 | TOST p=0.384 -> inconclusive
  C3_both   dAUC=-0.0002 p=0.967 | MDE(80% power)=0.0160 | TOST p=0.043 -> EQUIVALENT

Interpretation: no condition significantly beats counts; the design could detect a gain of ~0.016 AUC at 80% power, and the effect is statistically equivalent to zero within +/-0.01. The negative result is powered, not merely inconclusive.



05_llm_enrichment complete: 4-condition ablation + power/equivalence; 1 figure, 3 tables.
